# Reward delivery regression tests

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest
from baseline_rewards import TaskReward,MaxSupportReward
class Checks(unittest.TestCase):
    def test_high_water(self):
        t=TaskReward();self.assertEqual([t.calculate(s) for s in [0,1,1,0,2]],[0,1,0,0,1])
        t.reset();self.assertEqual(t.calculate(1),1)
        with self.assertRaises(ValueError):t.calculate(float('nan'))
    def test_fresh_identity_and_invalid(self):
        m=MaxSupportReward();self.assertEqual(m.calculate(None,False,False,None),0)
        self.assertEqual(m.calculate(.6,True,True,[1,1,2]),.6)
        self.assertEqual(m.calculate(.6,True,False,None),0)
        self.assertEqual(m.calculate(.6,True,True,[1,1,2]),0)
        self.assertEqual(m.calculate(.6,True,True,[1,2,3]),.6)
        m.reset();self.assertEqual(m.calculate(.6,True,True,[1,1,2]),.6)
        with self.assertRaises(ValueError):m.calculate(1.1,True,True,[1,3,4])
print('Reward delivery regression tests definitions/execution completed.')


Task progress and fresh-pair rewards definitions/execution completed.
Reward delivery regression tests definitions/execution completed.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_fresh_identity_and_invalid (__main__.Checks.test_fresh_identity_and_invalid) ... 

ok


test_high_water (__main__.Checks.test_high_water) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


Tests run: 2
